# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

#### Question 1 Part 1

In [6]:
from pyspark.sql.functions import monotonically_increasing_id

df_trips = df_trips.withColumn(
    "trip_id",
    monotonically_increasing_id()
)

df_trips.select("trip_id", "tpep_pickup_datetime",
                "tpep_dropoff_datetime",
                "passenger_count",
                "trip_distance").show()

+-----------+--------------------+---------------------+---------------+-------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|
+-----------+--------------------+---------------------+---------------+-------------+
|25769803776| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|
|25769803777| 2019-01-01 00:59:47|  2019-01-01 01:18:59|            1.0|          2.6|
|25769803778| 2018-12-21 13:48:30|  2018-12-21 13:52:40|            3.0|          0.0|
|25769803779| 2018-11-28 15:52:25|  2018-11-28 15:55:45|            5.0|          0.0|
|25769803780| 2018-11-28 15:56:57|  2018-11-28 15:58:33|            5.0|          0.0|
|25769803781| 2018-11-28 16:25:49|  2018-11-28 16:28:26|            5.0|          0.0|
|25769803782| 2018-11-28 16:29:37|  2018-11-28 16:33:43|            5.0|          0.0|
|25769803783| 2019-01-01 00:21:28|  2019-01-01 00:28:37|            1.0|          1.3|
|25769803784| 2019-01-01 00:32:01|  2019-01

#### Question 2 Part 1

In [7]:
df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance"
).orderBy(
    "passenger_count",
    ascending=False
).show(1)

+-----------+--------------------+---------------------+---------------+-------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|
+-----------+--------------------+---------------------+---------------+-------------+
|25770753732| 2019-01-05 13:12:29|  2019-01-05 13:12:32|            9.0|          0.0|
+-----------+--------------------+---------------------+---------------+-------------+
only showing top 1 row


#### Question 3 Part 1

In [8]:
from pyspark.sql.functions import avg

df_trips.select(
    avg("passenger_count").alias("average_passenger_count")
).show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



#### Question 4 Part 1

In [11]:
from pyspark.sql.functions import unix_timestamp

df_trips = df_trips.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp("tpep_dropoff_datetime")
        - unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

# Shortest trip by time
df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes"
).orderBy(
    "trip_duration_minutes",
    ascending=True
).show(1)

print("\n")

# Longest trip by time
df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes"
).orderBy(
    "trip_duration_minutes",
    ascending=False
).show(1)

+-----------+--------------------+---------------------+---------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+-----------+--------------------+---------------------+---------------------+
|25771006960| 2019-01-06 15:15:08|  2018-11-09 02:34:38|             -84280.5|
+-----------+--------------------+---------------------+---------------------+
only showing top 1 row


+-----------+--------------------+---------------------+---------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+-----------+--------------------+---------------------+---------------------+
|25769872043| 2019-01-01 07:01:20|  2019-01-31 14:29:21|    43648.01666666667|
+-----------+--------------------+---------------------+---------------------+
only showing top 1 row


#### Question 5 Part 1

In [13]:
from pyspark.sql.functions import to_date, count, month, year

trips_by_day = df_trips.filter(
    (year("tpep_pickup_datetime") == 2019) &
    (month("tpep_pickup_datetime") == 1)
).withColumn(
    "pickup_date",
    to_date("tpep_pickup_datetime")
).groupBy(
    "pickup_date"
).agg(
    count("*").alias("number_of_trips")
)

# Busiest day
trips_by_day.orderBy(
    "number_of_trips",
    ascending=False
).show(1)

print("\n")

# Slowest day
trips_by_day.orderBy(
    "number_of_trips",
    ascending=True
).show(1)

+-----------+---------------+
|pickup_date|number_of_trips|
+-----------+---------------+
| 2019-01-25|         292499|
+-----------+---------------+
only showing top 1 row


+-----------+---------------+
|pickup_date|number_of_trips|
+-----------+---------------+
| 2019-01-01|         189432|
+-----------+---------------+
only showing top 1 row


#### Question 6 Part 1

In [14]:
from pyspark.sql.functions import hour, count

trips_by_hour = df_trips.filter(
    (year("tpep_pickup_datetime") == 2019) &
    (month("tpep_pickup_datetime") == 1)
).withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
).groupBy(
    "pickup_hour"
).agg(
    count("*").alias("number_of_trips")
)

# Busiest hour
trips_by_hour.orderBy(
    "number_of_trips",
    ascending=False
).show(1)

print("\n")

# Slowest hour
trips_by_hour.orderBy(
    "number_of_trips",
    ascending=True
).show(1)

+-----------+---------------+
|pickup_hour|number_of_trips|
+-----------+---------------+
|         18|         515374|
+-----------+---------------+
only showing top 1 row


+-----------+---------------+
|pickup_hour|number_of_trips|
+-----------+---------------+
|          4|          61423|
+-----------+---------------+
only showing top 1 row


#### Question 7 Part 1

In [18]:
from pyspark.sql.functions import dayofweek, to_date, count, avg

trips_per_day = df_trips.filter(
    (year("tpep_pickup_datetime") == 2019) &
    (month("tpep_pickup_datetime") == 1)
).withColumn(
    "pickup_date",
    to_date("tpep_pickup_datetime")
).withColumn(
    "day_of_week",
    dayofweek("tpep_pickup_datetime")
).groupBy(
    "pickup_date",
    "day_of_week"
).agg(
    count("*").alias("number_of_trips")
)

# Average number of trips by day of week
average_by_weekday = trips_per_day.groupBy(
    "day_of_week"
).agg(
    avg("number_of_trips").alias("average_trips")
)

print("Decreasing Order")
average_by_weekday.orderBy(
    "average_trips",
    ascending=False
).show()

print("\n")
print("Creasing Order")
average_by_weekday.orderBy(
    "average_trips",
    ascending=True
).show()

Decreasing Order
+-----------+-------------+
|day_of_week|average_trips|
+-----------+-------------+
|          6|     271787.5|
|          5|     271398.4|
|          4|     253045.8|
|          7|    252494.75|
|          3|     241815.2|
|          2|     226941.0|
|          1|     214972.5|
+-----------+-------------+



Creasing Order
+-----------+-------------+
|day_of_week|average_trips|
+-----------+-------------+
|          1|     214972.5|
|          2|     226941.0|
|          3|     241815.2|
|          7|    252494.75|
|          4|     253045.8|
|          5|     271398.4|
|          6|     271787.5|
+-----------+-------------+



#### Question 8 Part 1

In [19]:
from pyspark.sql.functions import corr

# Correlation between trip distance and tip amount
df_trips.select(
    corr("trip_distance", "tip_amount").alias("distance_tip_correlation")
).show()

print("\n")

# Correlation between passenger count and tip amount
df_trips.select(
    corr("passenger_count", "tip_amount").alias("passenger_tip_correlation")
).show()

+------------------------+
|distance_tip_correlation|
+------------------------+
|      0.5269200663652668|
+------------------------+



+-------------------------+
|passenger_tip_correlation|
+-------------------------+
|     0.001084223312167...|
+-------------------------+



#### Question 9 Part 1

In [20]:
# Trip with the highest extra charge
df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "extra"
).orderBy(
    "extra",
    ascending=False
).show(1)

+-----------+--------------------+---------------------+------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime| extra|
+-----------+--------------------+---------------------+------+
|25775127259| 2019-01-23 08:58:09|  2019-01-23 08:58:09|535.38|
+-----------+--------------------+---------------------+------+
only showing top 1 row


#### Question 10 Part 1

In [23]:
# Look for strange / extreme values

# Extreme trip distances
df_trips.select(
    "trip_id",
    "trip_distance"
).orderBy(
    "trip_distance",
    ascending=False
).show(10)

print("\n")

# Extreme trip durations
df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes"
).orderBy(
    "trip_duration_minutes",
    ascending=False
).show(10)

print("\n")

# Negative trip durations
df_trips.filter(
    df_trips.trip_duration_minutes < 0
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes"
).show(10)

+-----------+-------------+
|    trip_id|trip_distance|
+-----------+-------------+
|25775877867|        831.8|
|25774090409|        700.7|
|25776574761|       214.01|
|25774511310|       211.36|
|25774685561|       201.27|
|25774617111|       160.52|
|25772371219|        144.2|
|25774680195|       143.63|
|25770948715|       142.88|
|25774715090|        132.8|
+-----------+-------------+
only showing top 10 rows


+-----------+--------------------+---------------------+---------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+-----------+--------------------+---------------------+---------------------+
|25769872043| 2019-01-01 07:01:20|  2019-01-31 14:29:21|    43648.01666666667|
|25770396038| 2019-01-03 22:24:36|  2019-01-27 10:41:17|   33856.683333333334|
|25770679632| 2019-01-05 04:21:40|  2019-01-27 01:53:46|              31532.1|
|25773519501| 2019-01-16 15:31:14|  2019-01-21 15:22:08|               7190.9|
|25771519041| 2019-01-08 20:

### Outlier analysis

Some values seem unusual in the dataset. For example, one trip is over 800 miles, which is very high for a NYC taxi trip.

There are also some incorrect trip durations, such as negative durations or trips lasting several days.

These values are probably data errors and could affect the analysis.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [24]:
# Load taxi zone lookup data

zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

response = requests.get(zone_url)

with open("/home/jovyan/work/taxi_zone_lookup.csv", "wb") as f:
    f.write(response.content)

df_zones = spark.read.csv(
    "/home/jovyan/work/taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

df_zones.show(10)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows


#### Question 1 Part 2

In [26]:
print("Pickups by borough")

pickup_boroughs = df_trips.join(
    df_zones,
    df_trips.PULocationID == df_zones.LocationID,
    "left"
)

pickup_boroughs.groupBy(
    "Borough"
).count().orderBy(
    "count",
    ascending=False
).show()

print("Dropoffs by borough")

dropoff_boroughs = df_trips.join(
    df_zones,
    df_trips.DOLocationID == df_zones.LocationID,
    "left"
)

dropoff_boroughs.groupBy(
    "Borough"
).count().orderBy(
    "count",
    ascending=False
).show()

Pickups by borough
+-------------+-------+
|      Borough|  count|
+-------------+-------+
|    Manhattan|6950965|
|       Queens| 471173|
|      Unknown| 159815|
|     Brooklyn|  91905|
|        Bronx|  18062|
|          N/A|   3890|
|          EWR|    446|
|Staten Island|    361|
+-------------+-------+

Dropoffs by borough
+-------------+-------+
|      Borough|  count|
+-------------+-------+
|    Manhattan|6817355|
|       Queens| 340972|
|     Brooklyn| 301105|
|      Unknown| 149097|
|        Bronx|  58085|
|          N/A|  16904|
|          EWR|  10914|
|Staten Island|   2185|
+-------------+-------+



#### Question 2 Part 2

In [29]:
from pyspark.sql.functions import hour

print("Borough hours")

borough_by_hour = pickup_boroughs.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
)

borough_hour_counts = borough_by_hour.groupBy(
    "Borough",
    "pickup_hour"
).count()

borough_hour_counts.orderBy(
    "count",
    ascending=False
).show(30)

print("\n")

borough_hour_counts.orderBy(
    "count",
    ascending=True
).show(30)

Borough hours
+---------+-----------+------+
|  Borough|pickup_hour| count|
+---------+-----------+------+
|Manhattan|         18|471539|
|Manhattan|         19|432836|
|Manhattan|         17|426498|
|Manhattan|         15|409119|
|Manhattan|         14|391823|
|Manhattan|         20|382357|
|Manhattan|         16|377245|
|Manhattan|         21|367636|
|Manhattan|         12|367189|
|Manhattan|         13|366911|
|Manhattan|         11|346110|
|Manhattan|          8|338274|
|Manhattan|          9|333426|
|Manhattan|         22|329592|
|Manhattan|         10|327978|
|Manhattan|          7|272314|
|Manhattan|         23|248416|
|Manhattan|          0|183353|
|Manhattan|          6|155963|
|Manhattan|          1|134734|
|Manhattan|          2|100363|
|Manhattan|          3| 71067|
|Manhattan|          5| 62775|
|Manhattan|          4| 53447|
|   Queens|         16| 29885|
|   Queens|         21| 29438|
|   Queens|         15| 28937|
|   Queens|         20| 28080|
|   Queens|         19| 2

#### Question 3 Part 2

In [30]:
from pyspark.sql.functions import dayofweek

print("Busiest days of the week by borough :")

borough_by_day = pickup_boroughs.withColumn(
    "day_of_week",
    dayofweek("tpep_pickup_datetime")
)

borough_day_counts = borough_by_day.groupBy(
    "Borough",
    "day_of_week"
).count()

borough_day_counts.orderBy(
    "count",
    ascending=False
).show(30)

Busiest days of the week by borough :
+---------+-----------+-------+
|  Borough|day_of_week|  count|
+---------+-----------+-------+
|Manhattan|          5|1229554|
|Manhattan|          4|1144782|
|Manhattan|          3|1086202|
|Manhattan|          6| 984950|
|Manhattan|          7| 927504|
|Manhattan|          2| 807748|
|Manhattan|          1| 770225|
|   Queens|          5|  78972|
|   Queens|          3|  78684|
|   Queens|          4|  75831|
|   Queens|          2|  66673|
|   Queens|          6|  64162|
|   Queens|          1|  59193|
|   Queens|          7|  47658|
|  Unknown|          5|  28929|
|  Unknown|          4|  25781|
|  Unknown|          3|  24523|
|  Unknown|          6|  21764|
|  Unknown|          2|  21417|
|  Unknown|          7|  20705|
|  Unknown|          1|  16696|
| Brooklyn|          3|  15779|
| Brooklyn|          5|  15714|
| Brooklyn|          4|  15101|
| Brooklyn|          6|  13092|
| Brooklyn|          7|  11604|
| Brooklyn|          1|  11099|
| 

#### Question 4 Part 2

In [31]:
from pyspark.sql.functions import avg

print("Average trip distance by borough :")

pickup_boroughs.groupBy(
    "Borough"
).agg(
    avg("trip_distance").alias("average_trip_distance")
).orderBy(
    "average_trip_distance",
    ascending=False
).show()

Average trip distance by borough :
+-------------+---------------------+
|      Borough|average_trip_distance|
+-------------+---------------------+
|Staten Island|   12.503601108033246|
|       Queens|   11.283218499361993|
|        Bronx|    7.233194552098303|
|     Brooklyn|    4.787677275447492|
|          N/A|    3.193850899742941|
|          EWR|    2.641098654708519|
|      Unknown|    2.415464130400774|
|    Manhattan|   2.2286693358402596|
+-------------+---------------------+



#### Question 5 Part 2

In [32]:

print("Average trip fare by borough :")
pickup_boroughs.groupBy(
    "Borough"
).agg(
    avg("fare_amount").alias("average_fare")
).orderBy(
    "average_fare",
    ascending=False
).show()

Average trip fare by borough :
+-------------+------------------+
|      Borough|      average_fare|
+-------------+------------------+
|          EWR| 76.24024663677126|
|          N/A|  59.5731593830335|
|Staten Island|45.289861495844896|
|       Queens| 35.14462651722029|
|        Bronx| 26.26890543682963|
|     Brooklyn|18.649132800172286|
|      Unknown|14.944423051653523|
|    Manhattan|10.792468572351568|
+-------------+------------------+



#### Question 6 Part 2

In [33]:
print("Highest fare :")
pickup_boroughs.select(
    "trip_id",
    "Borough",
    "Zone",
    "fare_amount",
    "trip_distance"
).orderBy(
    "fare_amount",
    ascending=False
).show(1)

print("\n")
print("Lowest fare")
pickup_boroughs.select(
    "trip_id",
    "Borough",
    "Zone",
    "fare_amount",
    "trip_distance"
).orderBy(
    "fare_amount",
    ascending=True
).show(1)

Highest fare :
+-----------+---------+--------------------+-----------+-------------+
|    trip_id|  Borough|                Zone|fare_amount|trip_distance|
+-----------+---------+--------------------+-----------+-------------+
|25772303431|Manhattan|Upper East Side S...|  623259.86|          2.4|
+-----------+---------+--------------------+-----------+-------------+
only showing top 1 row


Lowest fare
+-----------+-------+-----------+-----------+-------------+
|    trip_id|Borough|       Zone|fare_amount|trip_distance|
+-----------+-------+-----------+-----------+-------------+
|25774694425| Queens|JFK Airport|     -362.0|          0.0|
+-----------+-------+-----------+-----------+-------------+
only showing top 1 row


#### Question 7 Part 2

In [34]:

# We download the January 2025 taxi data
download_url_2025 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"

response = requests.get(download_url_2025)

jan_2025_trip_data = "/home/jovyan/work/yellow_tripdata_2025-01.parquet"

if response.status_code == 200:
    with open(jan_2025_trip_data, "wb") as f:
        f.write(response.content)

# We create a 2025 dataframe

df_trips_2025 = spark.read.parquet(jan_2025_trip_data)

df_trips_2025.show(5)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

#### Question 8 Part 2 (Year Comparaison)

In [35]:
from pyspark.sql.functions import avg

print("2019 averages")

df_trips.select(
    avg("passenger_count").alias("avg_passengers"),
    avg("trip_distance").alias("avg_distance"),
    avg("fare_amount").alias("avg_fare"),
    avg("tip_amount").alias("avg_tip")
).show()

print("\n")
print("\n2025 averages")

df_trips_2025.select(
    avg("passenger_count").alias("avg_passengers"),
    avg("trip_distance").alias("avg_distance"),
    avg("fare_amount").alias("avg_fare"),
    avg("tip_amount").alias("avg_tip")
).show()

2019 averages
+------------------+------------------+-----------------+------------------+
|    avg_passengers|      avg_distance|         avg_fare|           avg_tip|
+------------------+------------------+-----------------+------------------+
|1.5670317144945614|2.8301461681153532|12.52967677747685|1.8208300763883147|
+------------------+------------------+-----------------+------------------+




2025 averages
+------------------+-----------------+-----------------+------------------+
|    avg_passengers|     avg_distance|         avg_fare|           avg_tip|
+------------------+-----------------+-----------------+------------------+
|1.2978589658806226|5.855126178843539|17.08180276045484|2.9598127862758044|
+------------------+-----------------+-----------------+------------------+



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

#### Step 1 Part 3

In [36]:
# Create SQL temporary views
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

#### Step 2 Part 3

In [37]:
spark.sql("""
SELECT AVG(passenger_count) AS average_passenger_count
FROM trips
""").show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



#### Step 3 Part 3

In [38]:
spark.sql("""
SELECT *
FROM trips
ORDER BY passenger_count DESC
LIMIT 1
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+---------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    trip_id|trip_duration_minutes|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+---------------------+
|       2| 2019-01-05 13:12:29|  2019-01-05 13:12:32|            9.0|          0.0|  

#### Step 4 Part 3

In [40]:
spark.sql("""
SELECT 
    z.Borough,
    COUNT(*) AS number_of_pickups
FROM trips t
JOIN zones z
    ON t.PULocationID = z.LocationID
GROUP BY z.Borough
ORDER BY number_of_pickups DESC
LIMIT 1
""").show()

+---------+-----------------+
|  Borough|number_of_pickups|
+---------+-----------------+
|Manhattan|          6950965|
+---------+-----------------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing